# 5. PAR Model — Classical Yule-Walker


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Concepts

An autoregressive model uses previous observations to explain the current observation. A **Periodic Autoregressive (PAR)** model allows relationships to vary according to position in a repeating period.

### Analogy
Traffic may behave differently on Monday morning and Sunday afternoon. A PAR model can allow the rule to depend on the position in the cycle. The classical Yule-Walker approach estimates autoregressive relationships from autocovariance information.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import yule_walker
import matplotlib.pyplot as plt

# === INPUT: residual series from previous codifference steps ===
y = residual_series.values
index = residual_series.index

# === AIC-based selection ===
def compute_par_aic(y, p, L):
    n = len(y)
    phase_data = {s: [] for s in range(p)}
    total_residuals = []
    k_total = 0

    for t in range(p * L, n):
        phase = t % p
        lags = y[t - L:t][::-1]
        target = y[t]
        phase_data[phase].append((lags, target))

    for s in range(p):
        data_s = phase_data[s]
        if len(data_s) <= L:
            continue
        targets = np.array([target for _, target in data_s])
        if len(targets) < L + 1:
            continue
        try:
            rho, sigma = yule_walker(targets, order=L)
            k_total += L
            preds = []
            for lags, actual in data_s:
                if len(lags) == L:
                    pred = np.dot(rho, lags)
                    preds.append(actual - pred)
            total_residuals.extend(preds)
        except:
            continue

    rss = np.sum(np.square(total_residuals))
    T = len(total_residuals)
    if T == 0 or rss == 0:
        return np.inf
    aic = T * np.log(rss / T) + 2 * k_total
    return aic

# === Search best (p, L) ===
best_aic = np.inf
best_p = None
best_L = None

for p in range(2, 31, 2):
    for L in range(1, 4):
        aic = compute_par_aic(y, p, L)
        if aic < best_aic:
            best_aic = aic
            best_p = p
            best_L = L

print(f"Best Period (p): {best_p}")
print(f"Best Order (L): {best_L}")
print(f"Minimum AIC: {best_aic:.2f}")

# === Fit PAR(best_p, best_L) ===
phi = {}
phase_data = {s: [] for s in range(best_p)}
for t in range(best_p * best_L, len(y)):
    phase = t % best_p
    lags = y[t - best_L:t][::-1]
    target = y[t]
    phase_data[phase].append((lags, target))

for s in range(best_p):
    data_s = phase_data[s]
    targets = np.array([target for _, target in data_s])
    try:
        rho, sigma = yule_walker(targets, order=best_L)
        phi[s] = rho
    except:
        phi[s] = np.zeros(best_L)

# === Predict using fitted model ===
fitted = np.zeros_like(y)
for t in range(best_p * best_L, len(y)):
    phase = t % best_p
    coefs = phi.get(phase, np.zeros(best_L))
    lag_vec = y[t - best_L:t][::-1]
    fitted[t] = np.dot(coefs, lag_vec)

# === Plot Actual vs Predicted ===
plt.figure(figsize=(12, 4))
plt.plot(index, y, label='Actual Residuals', alpha=0.6)
plt.plot(index, fitted, label=f'PAR({best_L}) Prediction (p={best_p})', color='red')
plt.title('PAR Model Fit Using Yule-Walker')
plt.xlabel('Time')
plt.ylabel('Residual Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

